# PDDL Copilot — Results Analysis

Visualize experiment results from `results/` directory.

Reproduces Table 1 and chain evaluation plots from [Benyamin et al., 2025](https://arxiv.org/abs/2509.12987).

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")

## Load Results

Loads the most recent `single_task_*.json` and `chain_*.json` files.

In [ ]:
# Find the most recent result files
single_files = sorted(RESULTS_DIR.glob("single_task_*.json"))
chain_files = sorted(RESULTS_DIR.glob("chain_*.json"))

if not single_files:
    raise FileNotFoundError(
        f"No single_task_*.json found in {RESULTS_DIR}/. "
        "Run experiments first using run_experiment.py or experiment_notebook.ipynb."
    )

single_path = single_files[-1]
print(f"Loading: {single_path}")
single_data = json.loads(single_path.read_text())
df = pd.DataFrame(single_data)
print(f"  {len(df)} results loaded")

chain_data = None
if chain_files:
    chain_path = chain_files[-1]
    print(f"Loading: {chain_path}")
    chain_data = json.loads(chain_path.read_text())
    chain_df = pd.DataFrame(chain_data)
    print(f"  {len(chain_df)} chain results loaded")

## Table 1: Single-Task Success Rates

Pivot table showing success rate per model, condition (with/without tools), and task.

In [ ]:
df["condition"] = df["with_tools"].map({True: "tools", False: "no-tools"})

table1 = df.pivot_table(
    values="success",
    index=["model", "condition"],
    columns="task",
    aggfunc="mean",
).round(2)

table1

## Bar Chart: With Tools vs Without Tools

In [ ]:
models = df["model"].unique()
fig, axes = plt.subplots(1, len(models), figsize=(7 * len(models), 5), sharey=True)
if len(models) == 1:
    axes = [axes]

tasks = sorted(df["task"].unique())
x = range(len(tasks))
width = 0.35

for ax, model in zip(axes, models):
    mdf = df[df["model"] == model]
    tools_rates = [mdf[(mdf["task"] == t) & (mdf["with_tools"])]["success"].mean() for t in tasks]
    no_tools_rates = [mdf[(mdf["task"] == t) & (~mdf["with_tools"])]["success"].mean() for t in tasks]

    ax.bar([i - width / 2 for i in x], tools_rates, width, label="With Tools", color="#4C72B0")
    ax.bar([i + width / 2 for i in x], no_tools_rates, width, label="Without Tools", color="#DD8452")

    ax.set_title(model)
    ax.set_xticks(list(x))
    ax.set_xticklabels(tasks, rotation=45, ha="right")
    ax.set_ylabel("Success Rate")
    ax.set_ylim(0, 1.05)
    ax.legend()

plt.tight_layout()
plt.show()

## Chain Success Rate (if available)

Success rate vs chain length — shows how quickly performance degrades as task chains get longer.

In [ ]:
if chain_data:
    fig, ax = plt.subplots(figsize=(8, 5))
    for model in chain_df["model"].unique():
        mdf = chain_df[chain_df["model"] == model].sort_values("chain_length")
        ax.plot(mdf["chain_length"], mdf["success_rate"], marker="o", label=model)

    ax.set_xlabel("Chain Length (n)")
    ax.set_ylabel("Success Rate")
    ax.set_title("Multi-Task Chain Success Rate")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No chain results found. Run experiments with --chains to generate.")

## Per-Domain Breakdown

In [ ]:
domain_table = df.pivot_table(
    values="success",
    index=["domain_name", "condition"],
    columns="task",
    aggfunc="mean",
).round(2)

domain_table